# Expert Systems

## 📚 Learning Objectives

By completing this notebook, you will:
- Design rule-based expert systems
- Implement simple inference engines

## 🔗 Where this fits

**Builds on:** Unit 1, lesson 10 "Knowledge Representation and Reasoning" — facts and rules become a working inference engine.

**Used later in:** Course 02 (AIAT 112) — Unit 2, where forward and backward chaining are implemented in full.

---


## 🎯 The program that out-prescribed the specialists — and was never used

**MYCIN** was built at Stanford in the 1970s: about 600 rules for diagnosing
bacterial infections of the blood and the meninges, and recommending
antibiotics. In **1979** *JAMA* published a blinded evaluation (Yu et al.,
*Antimicrobial Selection by a Computer*). Eight independent infectious-disease
experts, who did not know which recommendations came from the machine, rated the
therapy chosen for ten real meningitis cases. **MYCIN's recommendations were
rated acceptable 65% of the time. The five faculty specialists scored between
42.5% and 62.5%.** The program beat every human it was compared against.

It was never deployed to a single patient. The obstacles were not accuracy: it
needed a mainframe at a time when hospitals had none, a clinician had to type a
long interactive consultation for each case, and nobody could answer who would be
liable when a program's advice harmed someone. Being right turned out not to be
sufficient.

**And then the commercial version of this idea did ship — and then collapsed.**
Digital Equipment Corporation ran **XCON** (originally R1, written by John
McDermott at Carnegie Mellon in 1978) to configure VAX computer orders. By the
mid-1980s it handled more than 90% of DEC's orders by value and DEC estimated it
saved about **$25 million a year**. The rule base grew past 10,000 rules, and by
1989 past 15,000. Maintaining it — every new product meant tracing interactions
across thousands of rules that could fire in any order — became more expensive
than the savings. That maintenance wall, repeated across the industry, is the
second AI winter you read about in Unit 1, notebook 02.

### What goes wrong without an explicit inference engine

You could write the diagnosis below as a nest of `if` statements. Then the
knowledge and the control flow are the same object: you cannot show a doctor what
the system believes, you cannot ask *why* it concluded flu, and adding one rule
means re-reading the whole function. Separating a **knowledge base** (facts and
rules, as data) from an **inference engine** (a loop that applies them) is what
buys you explanation, auditability and edit-by-a-non-programmer.


# Expert Systems

**Unit:** Unit 2: AI Concepts, Terminology, and Application Domains  

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand what expert systems are and their applications
- Learn the components of expert systems
- Distinguish between rule-based and ML-based systems
- Implement forward and backward reasoning
- Build a simple expert system using Python

---

## 1. Introduction to Expert Systems

**Definition:** Expert systems are AI programs that mimic the decision-making ability of a human expert in a specific domain.

**Components:**
- Knowledge Base: Contains domain-specific facts and rules
- Inference Engine: Applies rules to derive conclusions
- User Interface: Allows interaction with the system
- Explanation Facility: Explains how conclusions were reached


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Simple Expert System: Medical Diagnosis Assistant

class ExpertSystem:
    """Simple rule-based expert system"""
    
    def __init__(self):
        # Knowledge Base: Rules for medical diagnosis
        self.knowledge_base = {
            'fever': {
                'high': ['flu', 'infection'],
                'low': ['cold', 'allergy']
            },
            'cough': {
                'dry': ['flu', 'allergy'],
                'wet': ['cold', 'infection']
            },
            'headache': {
                'severe': ['flu', 'infection'],
                'mild': ['cold', 'allergy']
            }
        }
        
        # Rules for diagnosis
        self.rules = [
            {'conditions': {'fever': 'high', 'cough': 'dry', 'headache': 'severe'}, 
             'diagnosis': 'flu', 'confidence': 0.9},
            {'conditions': {'fever': 'high', 'cough': 'wet'}, 
             'diagnosis': 'infection', 'confidence': 0.85},
            {'conditions': {'fever': 'low', 'cough': 'dry'}, 
             'diagnosis': 'allergy', 'confidence': 0.75},
            {'conditions': {'fever': 'low', 'cough': 'wet'}, 
             'diagnosis': 'cold', 'confidence': 0.8}
        ]
    
    def forward_reasoning(self, symptoms):
        """
        Forward reasoning: Start with facts (symptoms) and derive conclusions (diagnosis)
        """
        matching_rules = []
        
        for rule in self.rules:
            # Check if all conditions in rule match symptoms
            if all(symptoms.get(key) == value 
                   for key, value in rule['conditions'].items() 
                   if key in symptoms):
                matching_rules.append(rule)
        
        if matching_rules:
            # Return best match (highest confidence)
            best_match = max(matching_rules, key=lambda x: x['confidence'])
            return {
                'diagnosis': best_match['diagnosis'],
                'confidence': best_match['confidence'],
                'matched_rules': len(matching_rules)
            }
        return {'diagnosis': 'unknown', 'confidence': 0.0}
    
    def backward_reasoning(self, target_diagnosis):
        """
        Backward reasoning: Start with goal (diagnosis) and find required conditions
        """
        required_symptoms = []
        
        for rule in self.rules:
            if rule['diagnosis'] == target_diagnosis:
                required_symptoms.append(rule['conditions'])
        
        return required_symptoms

# Example usage
expert = ExpertSystem()

print("=== Forward Reasoning Example ===")
symptoms = {'fever': 'high', 'cough': 'dry', 'headache': 'severe'}
result = expert.forward_reasoning(symptoms)
print(f"Symptoms: {symptoms}")
print(f"Diagnosis: {result['diagnosis']}")
print(f"Confidence: {result['confidence']:.2%}")

print("\n=== Backward Reasoning Example ===")
target = 'flu'
required = expert.backward_reasoning(target)
print(f"To diagnose '{target}', you need:")
for i, conditions in enumerate(required, 1):
    print(f"  Option {i}: {conditions}")

=== Forward Reasoning Example ===
Symptoms: {'fever': 'high', 'cough': 'dry', 'headache': 'severe'}
Diagnosis: flu
Confidence: 90.00%

=== Backward Reasoning Example ===
To diagnose 'flu', you need:
  Option 1: {'fever': 'high', 'cough': 'dry', 'headache': 'severe'}


## 📊 Same knowledge base, two directions — read the two outputs together

The cell ran one inference engine twice, changing exactly one thing: **which end
of the rule it starts from.**

| direction | what you give it | what it gives back | printed result |
|---|---|---|---|
| **Forward** (data-driven) | the symptoms | the conclusion they support | `flu`, confidence **90%** |
| **Backward** (goal-driven) | the hypothesis `flu` | the evidence that would establish it | `{fever: high, cough: dry, headache: severe}` |

**The conclusion these two runs support:** the same rules serve two different
jobs. Forward chaining is what you want when the data arrives first and you do not
know what you are looking for — monitoring, alarms, diagnosis from a full workup.
Backward chaining is what you want when you have a hypothesis and a limited budget
for tests, because it tells you *which question to ask next*. A doctor with one
test left needs the second engine, not the first.

**What they do not support:** any claim about accuracy. Both runs used one patient
and rules written by hand. Nothing here was measured against an outcome — which is
precisely the gap Bayes' theorem, two notebooks from now, is built to close.


## 💬 Discuss

The system above diagnosed `flu` at **90% confidence** by forward reasoning, then
backward reasoning listed exactly which symptoms would be needed to reach that
same conclusion.

1. Where does 0.90 come from? Nobody measured it — it is a number the rule author
   typed. MYCIN had the same problem and invented "certainty factors" to manage
   it, a scheme later criticised for not being probabilities at all. **Would you
   show that 90% to a patient?** If not, what would you show instead?
2. MYCIN beat five specialists in a blinded trial and was never used. List what
   *else*, beyond accuracy, a clinical tool needs before deployment — then check
   your list against the diagnosis system above and mark what it lacks.
3. Rule systems did not disappear; they moved. A 2026 language-model agent is
   given a **tool schema** — a declared list of actions with typed arguments and
   preconditions — and an explicit failure path. Compare that schema to the
   knowledge base above: what is genuinely new, and what is XCON with better
   syntax? Where would the 10,000-rule maintenance wall reappear?


## ⚠️ Where this breaks

- **The knowledge-acquisition bottleneck is the whole story.** Every rule is typed
  by a human expert, and experts are expensive, disagree with each other, and
  cannot articulate much of what they know. This is the wall XCON hit at 10,000+
  rules and it is why the field turned to learning from data.
- **Rules interact combinatorially.** With *n* rules there are far more than *n*
  ways they can fire in sequence. Adding rule 501 can silently change what rules
  1-500 conclude, and there is no test suite short enough to catch it.
- **Exact matching means brittle behaviour.** Our engine needs the symptoms to
  match a rule's conditions. A patient with high fever, a *wet* cough and a severe
  headache falls through every rule and gets nothing — not a lower-confidence
  answer, *nothing*. Rule systems do not degrade gracefully; they stop.
- **The confidence numbers are decoration.** 0.90 was written by hand and is not
  calibrated against any outcome data. Two rules firing does not multiply into a
  meaningful joint confidence. **Use instead:** Bayes' theorem with measured base
  rates — the next notebook — when you need a number you can defend.
- **It cannot learn.** Every correction is a code change by a knowledge engineer.
  A system seeing a thousand cases a week learns nothing from any of them.
- **When a rule system is still the right answer:** when the rules are *given* —
  by law, regulation, tax code, safety standard or clinical protocol — rather than
  discovered. Then hand-written rules are not a weak substitute for a model; they
  are the correct implementation, because the requirement is to follow the rule
  exactly and be able to prove that you did.


## 📚 References

1. Shortliffe, E. H., & Buchanan, B. G. (1975). *A Model of Inexact Reasoning in Medicine* (MYCIN). Mathematical Biosciences, 23(3–4), 351–379.
2. Buchanan, B. G., & Shortliffe, E. H. (1984). *Rule-Based Expert Systems: The MYCIN Experiments of the Stanford Heuristic Programming Project*. Addison-Wesley.
3. d'Avila Garcez, A., & Lamb, L. C. (2020). *Neurosymbolic AI: The 3rd Wave*. Artificial Intelligence Review (2023). <https://arxiv.org/abs/2012.05876>